# ChiralFold Toy Dataset Demo

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/ChiralFold/blob/master/demos/ChiralFold_Toy_Dataset_Demo.ipynb)

This fast demo uses the packaged toy ubiquitin fragment and should complete in under two minutes on a standard Colab CPU runtime.

Workflows covered:
1. Audit the packaged toy PDB.
2. Predict a short D-peptide (`AFWKELDR`).
3. Enumerate a short L/D sequence (`AFK`).
4. Correct a synthetic AF3-mimetic inverted D-alanine residue.

In [ ]:
# Cell 1 - Install the current ChiralFold release from GitHub.
!pip -q install "chiralfold @ git+https://github.com/Tommaso-R-Marena/ChiralFold.git"

In [ ]:
# Cell 2 - Import the public API and locate the packaged toy PDB.
from importlib import resources
from pathlib import Path
import tempfile
import urllib.request

import chiralfold
from chiralfold import ChiralFold, audit_pdb, correct_af3_output, detect_chirality_violations, enumerate_diastereomers, format_report

print("ChiralFold version:", chiralfold.__version__)

def get_toy_pdb() -> Path:
    packaged = resources.files("chiralfold").joinpath("data/examples/toy_ubiquitin_fragment.pdb")
    if packaged.is_file():
        return Path(str(packaged))

    fallback = Path("toy_ubiquitin_fragment.pdb")
    url = "https://raw.githubusercontent.com/Tommaso-R-Marena/ChiralFold/master/chiralfold/data/examples/toy_ubiquitin_fragment.pdb"
    urllib.request.urlretrieve(url, fallback)
    return fallback

toy_pdb = get_toy_pdb()
print("Toy PDB:", toy_pdb)

In [ ]:
# Cell 3 - Audit the toy ubiquitin fragment.
report = audit_pdb(str(toy_pdb))
print(format_report(report))
print("Summary:", {
    "residues": report["n_residues"],
    "chirality_pct": report["chirality"]["pct_correct"],
    "rama_favored_pct": report["ramachandran"]["pct_favored"],
    "overall_score": report["overall_score"],
})

In [ ]:
# Cell 4 - Predict a short D-peptide and enumerate a toy sequence.
model = ChiralFold(n_conformers=3)
pred = model.predict("AFWKELDR")
print("D-peptide prediction:")
print("  sequence:", pred["sequence"])
print("  chirality:", pred["chirality_pattern"])
print("  violations:", pred["chirality_violations"], "of", pred["n_chiral_residues"])
print("  conformers:", pred.get("n_conformers", len(pred.get("conformers", []))))

print("\nTop AFK diastereomers:")
for row in enumerate_diastereomers("AFK", top_n=3, n_conformers=1, seed=7):
    print(f"  rank {row['rank']}: {row['chirality_pattern']} score={row['score']:.1f} valid={row['valid']}")

In [ ]:
# Cell 5 - Correct a synthetic AF3-mimetic inverted D-alanine residue.
synthetic_d_ala_inverted = """\
HETATM    1  N   DAL A   1       1.201   0.847   0.000  1.00  0.00           N
HETATM    2  CA  DAL A   1       0.000   0.000   0.000  1.00  0.00           C
HETATM    3  C   DAL A   1      -1.250   0.881   0.000  1.00  0.00           C
HETATM    4  O   DAL A   1      -1.200   2.095   0.000  1.00  0.00           O
HETATM    5  CB  DAL A   1       0.000  -0.500   1.200  1.00  0.00           C
END
"""

with tempfile.TemporaryDirectory() as tmp:
    before_path = Path(tmp) / "synthetic_af3_inverted_d_ala.pdb"
    after_path = Path(tmp) / "synthetic_af3_corrected_d_ala.pdb"
    before_path.write_text(synthetic_d_ala_inverted)

    before = detect_chirality_violations(str(before_path))
    result = correct_af3_output(str(before_path), str(after_path))

print("Before correction:", before["n_violations"], "violation(s)")
print("After correction:", result["after"]["n_violations"], "violation(s)")
print(result["summary"])